# Week 7, Day 1: The singular value decomposition: rotate, stretch, rotate

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w007/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Take a matrix apart

Compute the economy SVD of `A`. Check that `U @ np.diag(s) @ Vt` rebuilds `A` (store the result of `np.allclose` in `recon_ok`) and that the columns of `U` and the rows of `Vt` are orthonormal (`ortho_ok`).

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
A = rng.normal(size=(5, 3))
U, s, Vt = ...
recon_ok = ...
ortho_ok = ...
print("singular values:", s.round(3), recon_ok, ortho_ok)

In [ ]:
import numpy as np
def test_shapes():
    "economy SVD of a 5 by 3 matrix"
    assert U.shape == (5, 3) and s.shape == (3,) and Vt.shape == (3, 3)

def test_props():
    "recon_ok and ortho_ok are both True"
    assert recon_ok is True and ortho_ok is True

def test_pairs():
    "Av_i equals s_i u_i for every i"
    assert np.allclose(A @ Vt.T, U * s)

test_shapes()
test_props()
test_pairs()
print("All checks passed")

### Exercise: Check it numerically

Compute the eigenvalues and eigenvectors of $A^\top A$ with `eigh`, sort them in descending order, and store the square roots of the eigenvalues in `sv_from_eig` and the eigenvectors (as columns) in `V_from_eig`. They should match `s` and the rows of `Vt` up to sign.

In [ ]:
lam, Vec = np.linalg.eigh(A.T @ A)
sv_from_eig = ...
V_from_eig = ...
print(sv_from_eig.round(4), s.round(4))

In [ ]:
import numpy as np
def test_sv():
    "sv_from_eig equals the singular values, in descending order"
    assert np.allclose(sv_from_eig, s)

def test_vecs():
    "each column of V_from_eig is plus or minus the matching right singular vector"
    assert np.allclose(np.abs(np.sum(V_from_eig * Vt.T, axis=0)), 1)

test_sv()
test_vecs()
print("All checks passed")

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
nA, nB = 30, 20
in_pat = np.linalg.qr(rng.normal(size=(nA, 2)))[0]      # two perpendicular input patterns in A
out_pat = np.linalg.qr(rng.normal(size=(nB, 2)))[0]     # two output patterns in B
Wab = 3.0 * np.outer(out_pat[:, 0], in_pat[:, 0]) + 1.5 * np.outer(out_pat[:, 1], in_pat[:, 1]) + 0.02 * rng.normal(size=(nB, nA))
print("shape", Wab.shape, " fraction of nonzero weights:", np.mean(Wab != 0))
plt.figure(figsize=(5, 3)); plt.imshow(Wab, cmap="RdBu_r"); plt.colorbar(); plt.xlabel("neuron in A"); plt.ylabel("neuron in B"); plt.show()

### Exercise: Find the transmitted patterns

Compute the singular values `sv` of `Wab`. Count how many exceed 10% of the largest (`eff_rank`). Then take the top right singular vector and measure how well it matches the first planted input pattern, `align` = their dot product. On average over random inputs, what share of the response in B passes through the top two modes? (It is the sum of their squared singular values over the total.)

In [ ]:
Uab, sv, Vtab = ...
eff_rank = ...
align = ...

share = np.sum(sv[:2] ** 2) / np.sum(sv ** 2)   # average share of B's response in the top two modes
plt.figure(figsize=(5, 3)); plt.semilogy(sv, "o"); plt.xlabel("index"); plt.ylabel("singular value"); plt.show()
print("effective rank", eff_rank, " alignment", round(align, 4), " share of output in top 2 modes:", round(share, 3))

In [ ]:
import numpy as np
def test_sv():
    "sv holds the singular values of Wab, largest first"
    assert np.allclose(sv, np.linalg.svd(Wab, compute_uv=False))

def test_rank():
    "eff_rank counts singular values above 10% of the largest: 2"
    assert eff_rank == 2

def test_align():
    "the top input direction found is the first planted input pattern (up to sign)"
    assert abs(align) > 0.99

test_sv()
test_rank()
test_align()
print("All checks passed")